# ⚡ BoneRAG — Free Google Colab GPU Backend (pyngrok)

### 📌 Hướng dẫn:
1. **Bật GPU**: `Runtime` → `Change runtime type` → `T4 GPU`
2. Điền **ngrok authtoken** vào ô `NGROK_TOKEN` ở Step 3
3. Bấm **Runtime → Run all (Ctrl+F9)**
4. Copy link `https://xxxx.ngrok-free.app` → dán vào Vercel `VITE_API_BASE_URL` → Redeploy

💡 *Mẹo: Dataset sẽ tự động lưu vào Google Drive ở lần đầu, từ lần sau mở Colab sẽ load cực nhanh trong 3 giây mà không cần tải lại!*

In [ ]:
# [Step 1] Kiểm tra GPU & Cài đặt thư viện
!nvidia-smi
!pip install -q torch torchvision transformers open_clip_torch faiss-cpu pillow numpy huggingface_hub pyngrok

In [ ]:
# [Step 2] Mount Google Drive & Tải mã nguồn + Dataset FracAtlas thật (4,082 ảnh X-quang)
import os
from pathlib import Path

# 1. Tự động Mount Google Drive để lưu dataset vĩnh viễn (load trong 3s ở các lần sau)
DRIVE_DIR = Path("/content/drive/MyDrive/BoneRAG_Data")
LOCAL_DATA_DIR = Path("/content/fracatlas_repo")
USE_DRIVE = False

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    USE_DRIVE = True
except Exception:
    print("ℹ️ Chạy môi trường ngoài Colab hoặc chưa cấp quyền Drive -> Dùng bộ nhớ tạm.")

# 2. Clone / Pull mã nguồn mới nhất
if not os.path.exists('/content/boneRAG'):
    !git clone https://github.com/thanhnghi-do-2k3/boneRAG.git /content/boneRAG
else:
    !cd /content/boneRAG && git pull
%cd /content/boneRAG

# 3. Tải hoặc giải nén 4,082 ảnh X-quang FracAtlas thật
drive_zip = DRIVE_DIR / "fracatlas_full.zip"
local_zip = Path("/content/fracatlas_full.zip")

has_local_jpgs = len(list(LOCAL_DATA_DIR.rglob("*.jpg"))) > 50 or len(list(LOCAL_DATA_DIR.rglob("*.jpeg"))) > 50
has_drive_zip = drive_zip.exists() if USE_DRIVE else False

if not has_local_jpgs:
    if has_drive_zip:
        print("⚡ ĐÃ TÌM THẤY DATASET TRONG GOOGLE DRIVE! Đang giải nén cực nhanh (3 giây)... ")
        !unzip -q -o "{drive_zip}" -d /content/fracatlas_repo || true
    else:
        target_zip = drive_zip if USE_DRIVE else local_zip
        print("📥 Đang tải bộ dữ liệu FracAtlas (4,082 ảnh X-quang) từ HuggingFace...")
        !wget -q --show-progress -O "{target_zip}" "https://huggingface.co/datasets/runananya/fracatlas/resolve/main/archive%20%289%29.zip" || curl -L -o "{target_zip}" "https://figshare.com/ndownloader/articles/22276042/versions/1"
        print("📦 Đang giải nén ảnh X-quang...")
        !unzip -q -o "{target_zip}" -d /content/fracatlas_repo || true
        if USE_DRIVE:
            print("💾 Đã lưu dữ liệu vào Google Drive! Lần sau bật Colab chỉ mất 3 giây解 nén.")

total_jpgs = len(list(LOCAL_DATA_DIR.rglob("*.jpg"))) + len(list(LOCAL_DATA_DIR.rglob("*.png")))
print(f'🎉 XÁC NHẬN: Đã có {total_jpgs} tệp ảnh X-quang y khoa FracAtlas thật trong ổ đĩa!')
print('✅ Mã nguồn & Bộ dữ liệu X-quang đã sẵn sàng!')

In [ ]:
# [Step 3] Đọc ngrok token từ Colab Secret (không hard-code token)
try:
    from google.colab import userdata
    NGROK_TOKEN = userdata.get("NGROK_AUTHTOKEN")
except Exception:
    NGROK_TOKEN = os.environ.get("NGROK_AUTHTOKEN", "")

import subprocess, time, urllib.request, os
from pyngrok import ngrok, conf

PORT = 8088
print(f'Đang giải phóng port {PORT}...')
subprocess.run(f'fuser -k {PORT}/tcp 2>/dev/null || true', shell=True)
subprocess.run('pkill -f \"demo-app/server.py\" 2>/dev/null || true', shell=True)
time.sleep(2)

if not NGROK_TOKEN:
    raise ValueError('Thiếu Colab Secret NGROK_AUTHTOKEN.')
conf.get_default().auth_token = NGROK_TOKEN

log_file = open('/content/bonerag_server.log', 'w')
server_proc = subprocess.Popen(
    ['python3', 'demo-app/server.py', '--host', '0.0.0.0', '--port', str(PORT), '--encoder', 'clip', '--generator', 'template'],
    stdout=log_file,
    stderr=log_file,
)

print(f'Đang chờ BoneRAG server khởi động (PID={server_proc.pid})...')
for attempt in range(40):
    time.sleep(3)
    if server_proc.poll() is not None:
        print('Server đã dừng. Log cuối:')
        print(Path('/content/bonerag_server.log').read_text()[-5000:])
        raise RuntimeError('Server crashed.')
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/', timeout=5) as response:
            if response.status == 200:
                print(f'Server sẵn sàng sau {(attempt + 1) * 3}s.')
                break
    except Exception:
        print(f'Đang chờ server... [{attempt + 1}/40]', end='\r')
else:
    print('Server timeout. Log cuối:')
    print(Path('/content/bonerag_server.log').read_text()[-5000:])
    raise RuntimeError('Server timeout.')


In [ ]:
# [Step 4] Mở ngrok tunnel & in ra Public URL (Dừng tức thì khi bấm Stop / Ctrl+M I)
import urllib.request

# Đóng tunnel cũ nếu có
try:
    ngrok.kill()
    time.sleep(1)
except Exception:
    pass

tunnel = ngrok.connect(PORT, bind_tls=True)
PUBLIC_URL = tunnel.public_url

print()
print('=' * 65)
print('🚀 BACKEND GPU ĐANG CHẠY — COPY LINK NÀY VÀO VERCEL:')
print('=' * 65)
print(f'  VITE_API_BASE_URL = {PUBLIC_URL}')
print('=' * 65)
print('📌 Sau khi dán vào Vercel → Save → Redeploy!')
print('⚠️  Giữ ô này đang chạy. Đóng ô này bằng nút ⏹️ Stop (hoặc Ctrl+M I).')

# Giữ ngrok tunnel sống (sử dụng sleep nhỏ 1s để dừng ngắt lập tức khi bấm Stop)
try:
    counter = 0
    while True:
        time.sleep(1)
        counter += 1
        if counter >= 30:
            counter = 0
            try:
                urllib.request.urlopen(f'http://127.0.0.1:{PORT}/api/health', timeout=5)
            except Exception:
                print('⚠️ Server không phản hồi!')
except (KeyboardInterrupt, SystemExit):
    print('
🛑 Đã dừng tunnel thành công!')
    try:
        ngrok.kill()
    except Exception:
        pass